# Working with structured data - Part 2

This notebook accompanies the script <strong><span style="color:red;">06_pandas_part_B.pdf</span></strong>  and provides practical examples related to its content.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42) 

<hr style="border: none; height: 20px; background-color: green;">

# 6. Hierarchical Indexing with MultiIndex

### Hierarchical Indexing - The "bad/tempting" way: tuples as keys
If we track both `state` and `year`, it is tempting to use tuples as the index.

In [ ]:
index = [
    ('California', 2000), ('California', 2010),
    ('New York', 2000), ('New York', 2010),
    ('Texas', 2000), ('Texas', 2010),
]
populations = [33871648, 37253956, 
               18976457, 19378102, 
               20851820, 25145561]

ser_state_population = pd.Series(populations, index=index)
ser_state_population

In [ ]:
# California 2010
ser_state_population[('California', 2010)]

In [ ]:
# Slicing the Series 
ser_state_population.loc[('New York', 2010):('Texas', 2010)]

In [ ]:
# All years for California
idx_states_2010 = [idx for idx in ser_state_population.index if idx[0] == 'California']
ser_state_population.loc[idx_states_2010]

<hr style="border: none; height: 10px; background-color: LightBlue;">

## A cleaner way: Pandas `MultiIndex`


In [ ]:
index = [
    ('California', 2000), ('California', 2010),
    ('New York', 2000), ('New York', 2010),
    ('Texas', 2000), ('Texas', 2010),
]
index = pd.MultiIndex.from_tuples(
    index,
    names=['state', 'year'] # helps label the hierarchy
    )
index

#### Create a multi-indexed `Series` and `DartaFrame`

In [ ]:
populations = [33871648, 37253956, 
               18976457, 19378102, 
               20851820, 25145561]


# Series (script sceenshots)
ser_state_population = pd.Series(populations, index=index)
ser_state_population

In [ ]:
# DataFrame
df_state_population = pd.DataFrame(populations, index=index, columns=['population'])
df_state_population

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Indexing and Selection
-  `.loc[]`
-  `.xs()`
- `.iloc[]`

In [ ]:
ser_state_population

<hr style="border: none; height: 5px; background-color: gray;">

### `.loc[]` (label-based)

- **IMPORTANT:** `.loc[]` assumes the label refers to the **outermost level**!
- If you want to access a level other than the outermost one, you must know its position and select it using appropriate indexing (e.g., with `:` in `.loc`).
- General label-based indexing
- Preserves all index levels

In [ ]:
# California 2010
ser_state_population.loc[('California', 2010)]

In [ ]:
# All years for California
ser_state_population.loc['California']

##### Position-based selection in `.loc` for **non-outer** levels

If you want to access a level other than the outermost one, you must know its position and select it using appropriate indexing (e.g., with `:` in `.loc`).

In [ ]:
# All states for 2010 -> Series
ser_state_population.loc[:, 2010]                

In [ ]:
# All states for 2010 -> DataFrame
# Syntax: df.loc[rows, columns]
df_state_population.loc[(slice(None), 2010), :]  

<hr style="border: none; height: 5px; background-color: gray;">

### Selecting with `.xs()`

- allows explicit selection by level
- cross-section for a specific level
- reduces the dimensionality (removes the selected level)

In [ ]:
ser_state_population.xs(2010, level='year')

In [ ]:
# Selecting by level index (position) is also possible, but less explicit than using level names
ser_state_population.xs(2010, level=1)  

In [ ]:
ser_state_population.xs('California', level='state')

In [ ]:
ser_state_population.xs(('California', 2010), 
                        level=['state', 'year'])

<hr style="border: none; height: 5px; background-color: gray;">

#### `.iloc[]` (position-based)

In [ ]:
# Row index 2
ser_state_population.iloc[2:5]

In [ ]:
# Last row
ser_state_population.iloc[-1]

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Slicing a MultiIndex

In [ ]:
ser_state_population = ser_state_population.sort_index()
ser_state_population.loc["California":"New York"]

<hr style="border: none; height: 5px; background-color: gray;">

### Same Series, but shuffled (unsorted MultiIndex)

In [ ]:
index = [
    ('Texas', 2010),
    ('California', 2000),
    ('New York', 2010),
    ('Texas', 2000),
    ('California', 2010),
    ('New York', 2000),
]

index = pd.MultiIndex.from_tuples(
    index,
    names=['state', 'year']
)

populations = [
    25145561,
    33871648,
    19378102,
    20851820,
    37253956,
    18976457,
]

ser_state_population_unsorted = pd.Series(populations, index=index)

ser_state_population_unsorted

#### MultiIndex slicing requires a sorted index

In [ ]:
from pandas.errors import UnsortedIndexError

try:
    ser_state_population_unsorted.loc["California":"New York"]
except UnsortedIndexError as e:
    # Pandas checks whether the index is lexsorted before range slicing.
    # If not sorted, slicing is ambiguous → raises UnsortedIndexError.
    print(type(e).__name__, e)

In [ ]:
ser_sorted = ser_state_population_unsorted.sort_index()
ser_sorted.loc["California":"New York"]

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Reshaping with `.unstack()` and `.stack()`
`.unstack()` turns one index level into columns, adding a new dimension.    
`.stack()` reverses that operation.

In [ ]:
ser_state_population

In [ ]:
ser_state_population.unstack()

<hr style="border: none; height: 10px; background-color: LightBlue;">

## MultiIndex for an existing DataFrame

Often you start with a regular DataFrame and then call `.set_index()`.

In [ ]:
df_raw = pd.read_csv('../data/csv/population_dataset.csv')
df_raw.head(10)

In [ ]:
df = df_raw.set_index(['State', 'Year'])
df.head(15)

In [ ]:
df.unstack()

<hr style="border: none; height: 10px; background-color: LightBlue;">

## MultiIndex from product

In [ ]:
states = ["California", "New York", "Texas"]
years = [2000, 2010]

index = pd.MultiIndex.from_product(
    [states, years],
    names=["State", "Year"]
)
index

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Creating a 5-level MultiIndex using the Cartesian product

This example demonstrates how to represent a five-level dataset using a MultiIndex in pandas, allowing structured querying and hierarchical organization

In [ ]:
countries = ['USA', 'Germany']
years = [2000, 2010]
genders = ['Male', 'Female']
age_groups = ['0-18', '19-65', '65+']
income_groups = ['Low', 'Medium', 'High']

multi_index = pd.MultiIndex.from_product(
    [countries, years, genders, age_groups, income_groups],
    names=['Country', 'Year', 'Gender', 'Age Group', 'Income Group']
)

df_5l = pd.DataFrame({
    'Population': np.random.randint(1_000, 100_000, size=len(multi_index)),
    'GDP': (np.random.lognormal(mean=11, sigma=0.5, size=len(multi_index))*1000).astype(int)
}, index=multi_index)

df_5l.head(12)

<hr style="border: none; height: 5px; background-color: gray;">

#### `.xs(level='...')` example in a 5d MultiIndex

In [ ]:
# Population of females aged 19–65 in Germany in 2000
df_5l.xs(('Germany', 2000, 'Female', '19-65'), level=['Country', 'Year', 'Gender', 'Age Group'])

<hr style="border: none; height: 5px; background-color: gray;">

### `.unstack()` / `.stack()` examples in a 5d MultiIndex

In [ ]:
df_5l.unstack().unstack().head(10)

In [ ]:
df_5l.unstack().unstack().stack().stack().head(10)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## `MultiIndex` for columns
Rows and columns are symmetric: both can have multiple index levels.

In [ ]:
index = pd.MultiIndex.from_product(
    [[2023, 2024], [1, 2]],
    names=['year', 'visit']
)

columns = pd.MultiIndex.from_product(
    [['Bob', 'Guido', 'Sue'], ['HR', 'Temp']],
    names=['subject', 'type']
)

# Mock some data
data = np.round(np.random.randn(4, 6), 1)
data[:, ::2] *= 10  # Scale HR values
data += 37          # Shift values to a reasonable range

# Create the DataFrame
health_data = pd.DataFrame(data, index=index, columns=columns)

health_data

<hr style="border: none; height: 20px; background-color: green;">

# 2. Working with Missing Data

Missing values are common in real-world datasets. Pandas supports several sentinel values:
- `None` (Python)
- `np.nan` (IEEE floating point NaN)
- `pd.NA` (Pandas nullable missing value)
- `pd.NaT` (missing timestamps)


## Representation vs. detection/selection

In [ ]:
df = pd.DataFrame({
    "A": [1, 2, None, 4],
    "B": [np.nan, 5, 6, 7],
    "C": [10, pd.NA, 12, 13],
    "D": [pd.NA, "text", "data", pd.NA]
})
print("\nDataFrame with Sentinel Values:")
df

In [ ]:
print("Boolean Mask (isna()):")
df.isna()

### Object dtype vs nullable Int64: performance and type consistency

#### dtypes

In [ ]:
print(f'Column A dtype: {df["A"].dtype}')
print(f'Column B dtype: {df["B"].dtype}')
print(f'Column C dtype: {df["C"].dtype}    <-- object dtype can be a performance and memory bottleneck for large Series')
print(f'Column D dtype: {df["D"].dtype}')

In [ ]:
df.info()

#### values, missing values and their types

In [ ]:
# values and their types
print(f'Column A existing value: {df.loc[0, "A"]},  {type(df.loc[0, "A"])}')
print(f'Column B existing value: {df.loc[1, "B"]},  {type(df.loc[1, "B"])}')
print(f'Column C existing value: {df.loc[0, "C"]},   {type(df.loc[0, "C"])}')
print(f'Column D existing value: {df.loc[1, "D"]}, {type(df.loc[1, "D"])} \n')

# missing values and their types
print(f'Column A missing value: {df.loc[2, "A"]},   {type(df.loc[0, "A"])}')
print(f'Column B missing value: {df.loc[0, "B"]},   {type(df.loc[0, "B"])}')
print(f'Column C missing value: {df.loc[1, "C"]},  {type(df.loc[1, "C"])}')
print(f'Column D missing value: {df.loc[0, "D"]},   {type(df.loc[0, "D"])}')

### Solution: Convert to integer dtype (Int64)

In [ ]:
df["C"] = df["C"].astype(pd.Int64Dtype())

print(f'Column C existing value: {df.loc[0, "C"]},   {type(df.loc[0, "C"])}')
print(f'Column C missing value: {df.loc[1, "C"]},  {type(df.loc[1, "C"])} \n')

df.info()

### Performance comparison on different object dtypes

In [ ]:
# Create Python list with missing values (every 10th element)
n = 1_000_000
data = list(range(n))
data[::10] = [None] * len(data[::10])

# Create Pandas Series with different dtype handling of missing values
ser_int = pd.Series(data, dtype=pd.Int64Dtype())   # nullable integer → uses pd.NA
ser_float = pd.Series(data)                        # inferred float64 → None converted to np.nan
ser_obj = pd.Series(data, dtype="object")          # object dtype     → keeps Python None

print(f"series int dtype:    {ser_int.dtype}")
print(f"series float dtype:  {ser_float.dtype}")
print(f"series object dtype: {ser_obj.dtype} \n")

# Inspect actual stored missing values
i = 0
print(f"series int missing:    {ser_int.iloc[i], type(ser_int.iloc[i])}") # <NA>
print(f"series float missing:  {ser_float.iloc[i], type(ser_float.iloc[i])}")   # nan
print(f"series object missing: {ser_obj.iloc[i], type(ser_obj.iloc[i])} \n") # None

# Inspect actual stored regular values
i = 1
print(f"series int value type:    {ser_int.iloc[i], type(ser_int.iloc[i])}")
print(f"series float value type:  {ser_float.iloc[i], type(ser_float.iloc[i])}") 
print(f"series object value type: {ser_obj.iloc[i], type(ser_obj.iloc[i])} \n")

# Performance comparison
%timeit ser_int.sum()    # optimized nullable integer operations
%timeit ser_float.sum()  # fast NumPy float operations
%timeit ser_obj.sum()    # slow: Python-level operations (object dtype)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## 'None'

#### The right way to check for `None`

In [ ]:
x = None
print(x is None) # Detection

#### `None` is not numeric

In [ ]:
try:
    1 + None
except TypeError as e:
    print("Unsupported operand")

#### Python list to pandas Series: None becomes NaN and upcasts to float64

In [ ]:
pd.Series([1, 2, None, 4])

#### Truthiness

In [ ]:
bool(None)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## `np.nan`

#### Basic properties

In [ ]:
print(type(np.nan))
print(np.nan == np.nan)
print(np.nan != np.nan)
print(pd.isna(np.nan))  # Detection

#### Aggregations: "normal" vs NaN-aware

In [ ]:
s = pd.Series([1.0, np.nan, 3.0])

print("sum: ", s.sum())                  # default skipna=True 
print("sum: ", s.sum(skipna=False))
print("mean:", s.mean())

#### Use `pd.isna()` to detect missing values

In [ ]:
pd.isna([None, np.nan, pd.NA, pd.NaT])

#### Truthiness

In [ ]:
bool(np.nan)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## `pd.NA`

#### Basic properties of pd.NA

three-valued logic

In [ ]:
print(f"pd.isna(pd.NA):  {pd.isna(pd.NA)}")  # Detection
print(f"pd.notna(pd.NA): {pd.notna(pd.NA)}")
print(f"pd.NA & True:    {pd.NA & True}")
print(f"pd.NA | False:   {pd.NA | False}")
print(f"~pd.NA:          {~pd.NA}")

#### pd.NA is the missing value for nullable dtypes (Int64, boolean, string, Float64)

In [ ]:
ser_int = pd.Series([1, pd.NA, 3], dtype=pd.Int64Dtype())
ser_bool = pd.Series([True, pd.NA, False], dtype=pd.BooleanDtype())
ser_str = pd.Series(["a", pd.NA, "c"], dtype=pd.StringDtype())

print(f"ser_int values:  {ser_int.tolist()}")
print(f"ser_int dtype:   {ser_int.dtype} \n")

print(f"ser_bool values: {ser_bool.tolist()}")
print(f"ser_bool dtype:  {ser_bool.dtype} \n")

print(f"ser_str values:  {ser_str.tolist()}")
print(f"ser_str dtype:   {ser_str.dtype}")

### Arithmetic propagates <NA> in nullable dtypes

In [ ]:
s = pd.Series([1, pd.NA, 3], dtype=pd.Int64Dtype())

print(s + 1)

#### Default dtype converts None to NaN vs explicit Int64 preserves pd.NA

In [ ]:
ser_obj = pd.Series([1, None, 3])          
ser_int = pd.Series([1, None, 3], dtype=pd.Int64Dtype())  

print("ser_obj dtype:", ser_obj.dtype)
print("ser_int dtype:", ser_int.dtype)

#### **Beware**: Casting a nullable integer (with `pd.NA`) to a **NumPy** integer dtype will fail.

In [ ]:
s = pd.Series([1, pd.NA, 3], dtype=pd.Int64Dtype())

try:
    print(s.astype(np.int64))# NumPy integer
except Exception as e:
    print("astype(np.int64) fails:", repr(e))

# You can fill missing values first, then cast
print(s.fillna(0).astype(pd.Int64Dtype()))

<hr style="border: none; height: 10px; background-color: LightBlue;">

## `pd.NaT`

####  Create pd.NaT and test it

In [ ]:
x = pd.NaT
print(x)
print(pd.isna(x)) # Detection

#### pd.NaT in a datetime Series (dtype becomes datetime64[ns])

In [ ]:
s = pd.Series([pd.Timestamp("2026-01-01"), 
               pd.NaT, 
               pd.Timestamp("2026-01-03")])
print(s)
print("isna:", s.isna().tolist())

#### Datetime arithmetic propagates NaT

In [ ]:
s = pd.Series([pd.Timestamp("2026-01-01"), pd.NaT])
print(s + pd.Timedelta(days=1))

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Detecting Missing Values

In [ ]:
df = pd.DataFrame(
    {
        "age": [34, None, 52, None],
        "income": [52000, np.nan, 61000, 45000],
        "joined": [pd.Timestamp("2024-01-01"), None, pd.Timestamp("2024-02-01"), None],
    },
    index=pd.Index(["cust_001", "cust_002", "cust_003", "cust_004"], name="customer_id"),
)
df

In [ ]:
# Detect missing values
mask = df.isna()
mask

In [ ]:
print("\nRows with any missing:")
df.loc[mask.any(axis=1)]

In [ ]:
# Rows with any missing values
mask.sum()

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Drop Missing Values

In [ ]:
df = pd.DataFrame(
    {
        "age": [34, None, 52, None],
        "income": [52000, np.nan, 61000, 45000],
        "joined": [pd.Timestamp("2024-01-01"), None, pd.Timestamp("2024-02-01"), None],
    },
    index=pd.Index(["cust_001", "cust_002", "cust_003", "cust_004"], name="customer_id"),
)
df

In [ ]:
# Drop rows that contain ANY missing value (complete cases only)
df.dropna()

In [ ]:
# Drop rows only if ALL values are missing (keeps partially observed rows)
df.dropna(how="all")

In [ ]:
# Drop rows based on a subset of columns (e.g., require age and income)
df.dropna(subset=["age", "income"])

In [ ]:
# Keeps only rows that have at least one non-missing value
df.dropna(thresh=1)

In [ ]:
# Keeps only columns that have at least 3 non-missing value
df.dropna(axis=1, thresh=3)

In [ ]:
# Drop Columns with Any Missing Values
df.dropna(axis=1, how="any")

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Replacing Missing Values

<hr style="border: none; height: 5px; background-color: gray;">

### 1) Imputation: Fixed-value & simple-statistic imputation

In [ ]:
df = pd.DataFrame(
    {"age": [34, None, 52, None],
     "income": [52000, np.nan, 61000, 45000],
     "city": ["ZH", None, "BE", None]},
    index=pd.Index(["cust_001", "cust_002", "cust_003", "cust_004"], name="customer_id"),
)
df

In [ ]:
df_imputed = df.copy()
df_imputed["age"] = df_imputed["age"].fillna(df_imputed["age"].mean())
df_imputed["income"] = df_imputed["income"].fillna(df_imputed["income"].median())
df_imputed["city"] = df_imputed["city"].fillna("UNKNOWN")

df_imputed

<hr style="border: none; height: 5px; background-color: gray;">

### 2. Imputation: Forward/Backward fill

In [ ]:
df_ts = pd.DataFrame(
    {
        "value": [1.0, np.nan, np.nan, 2.0, np.nan, np.nan, 3.0]
    },
    index=pd.date_range("2024-01-01", periods=7, freq="D"),
)
df_ts

In [ ]:
df_filled = df_ts.copy()
df_filled["ffill_limit1"] = df_filled["value"].ffill(limit=1)
df_filled["ffill_full"] = df_filled["value"].ffill()
df_filled["bfill"] = df_filled["value"].bfill()

df_filled

<hr style="border: none; height: 5px; background-color: gray;">

### 3. Imputation: Group-based imputation 

In [ ]:
df = pd.DataFrame(
    {
        "region": ["A", "A", "A", "B", "B", "B"],
        "income": [52000, np.nan, 58000, 90000, np.nan, 120000],
    }
)

In [ ]:
df_group_imputed = df.copy()
group_median = df_group_imputed.groupby("region")["income"].transform("median")
df_group_imputed["income_imputed"] = df_group_imputed["income"].fillna(group_median)

df_group_imputed

<hr style="border: none; height: 5px; background-color: gray;">

### 4. Imputation: Missingness as signal

In [ ]:
df = pd.DataFrame(
    {"age": [34, None, 52, None],
     "income": [52000, np.nan, 61000, 45000],
     "city": ["ZH", None, "BE", None]},
    index=pd.Index(["cust_001", "cust_002", "cust_003", "cust_004"], name="customer_id"),
)
df

In [ ]:
df_feat = df.copy()
df_feat["income_missing"] = df_feat["income"].isna()

df_feat

<hr style="border: none; height: 5px; background-color: gray;">

### 5) Imputation: Model-based imputation

In [ ]:
# income roughly correlates with age, experience, and spending
df = pd.DataFrame({
    "age":        [25, 30, 35, 40, 45],
    "income":     [50000, 60000, np.nan, 80000, np.nan],
    "experience": [2, 5, 7, 10, 12],
    "spending":   [2000, 2500, 3000, 4000, 4500],
})
df

In [ ]:
from sklearn.impute import KNNImputer

df_knn = pd.DataFrame(
    KNNImputer(n_neighbors=2).fit_transform(df),columns=df.columns
)

df_knn

<hr style="border: none; height: 5px; background-color: gray;">

### 6. Interpolation: order-based

In [ ]:
df_ts = pd.DataFrame(
    {
        "value": [1.0, np.nan, np.nan, 2.0, np.nan, 4.0]
    },
    index=pd.to_datetime(
        ["2024-01-01", "2024-01-02", "2024-01-04", "2024-01-07", "2024-01-08", "2024-01-10"]
    ),
)
df_ts

In [ ]:
df_ts_interpolated = df_ts.copy()
df_ts_interpolated["linear"] = df_ts_interpolated["value"].interpolate(method="linear") # linear, quadratic, cubic
df_ts_interpolated["time"] = df_ts_interpolated["value"].interpolate(method="time") 

df_ts_interpolated

<hr style="border: none; height: 20px; background-color: green;">

# 8. Concatenating Datasets
`pd.concat()` combines Series/DataFrames along an axis:
- `axis=0`: stack vertically (add rows)
- `axis=1`: stack horizontally (add columns)

#### Concatenations in Pandas

In [ ]:
df1 = pd.DataFrame(
    {"A": [1, 2], "B": [3, 4], "C": [5, 6]},
    index=[0, 1]
)

df2 = pd.DataFrame(
    {"A": [7, 8], "B": [9, 10]},
    index=[1, 2]
)

display(df1, df2)

#### Basic row-wise concatenation 

In [ ]:
pd.concat([df1, df2])

#### Basic column-wise Concatenation (`axis=1`)

In [ ]:
pd.concat([df1, df2], axis=1)

### Duplicate indices

Unlike NumPy’s np.concatenate(), pandas preserves indices even if they are duplicated.  

By setting verify_integrity=True, pd.concat() raises a ValueError if duplicate indices are detected in the result. This helps catch unintended index duplication early, preventing potential issues when working with index-based operations.


In [ ]:
try:
    pd.concat([df1, df2], verify_integrity=True)
except ValueError as e:
    print('ValueError:', e)

Reset indices during concatenation with `ignore_index=True`.

In [ ]:
pd.concat([df1, df2], ignore_index=True)

In [ ]:
pd.concat([df1, df2], ignore_index=True, axis=1)

Add a key level to create a `MultiIndex` in the result with `keys=`.

In [ ]:
pd.concat([df1, df2], keys=['x', 'y'])

Keep only the intersection of columns with `join='inner'` (avoids extra `NaN` columns).

In [ ]:
pd.concat([df1, df2], join='inner')

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Merging and Joining Datasets
`pd.merge()` / `df.merge()` combine tables by aligning rows on key columns (SQL-style joins).

In [ ]:
df_1 = pd.DataFrame({
    'employee': ['Bob', 'Jake', 'Lisa', 'Sue'],
    'group': ['Accounting', 'Engineering', 'Engineering', 'HR']
})

df_2 = pd.DataFrame({
    'employee': ['Lisa', 'Bob', 'Jake', 'Sue'],
    'hire_date': [2004, 2008, 2012, 2014]
})
display(df_1, df_2)

#### `pd.merge()` vs. `df.merge()` – Two Ways to Merge DataFrames
Both `pd.merge()` and `df.merge()` perform the same operation, merging two DataFrames based on a key


In [ ]:
pd.merge(df_1, df_2)

In [ ]:
df_1.merge(df_2)

<hr style="border: none; height: 5px; background-color: gray;">

### The `on` Option in `pd.merge()`

- The `on` parameter specifies the column(s) that should be used as the key(s) for merging
- It ensures that the merge operation is performed based on a common identifier between both DataFrames
- Without `on`, pandas tries to auto-detect common columns, but it's best to explicitly define them


In [ ]:
df_1.merge(df_2, on='employee')

<hr style="border: none; height: 5px; background-color: gray;">

### Many-to-many join
If both sides have duplicate keys, the result can expand (cartesian product per key).

In [ ]:
df_3 = pd.DataFrame({
    "employee": ["Bob", "Jake", "Lisa", "Sue"],
    "group": ["Accounting", "Engineering", "Engineering", "HR"],
    "hire_date": [2008, 2012, 2004, 2014]
})

df_4 = pd.DataFrame({
    "group": ["Accounting", "Engineering", "HR"],
    "supervisor": ["Carly", "Guido", "Steve"]
})

display(df_3, df_4)

In [ ]:
pd.merge(df_3, df_4, on='group')

<hr style="border: none; height: 5px; background-color: gray;">

#### Many-to-Many Joins
A Many-to-Many join occurs when both key columns in the merging DataFrames contain duplicate values


In [ ]:
df_6 = pd.DataFrame({
    "group": [
        "Accounting", "Accounting",
        "Engineering", "Engineering",
        "HR", "HR"
    ],
    "skills": [
        "math", "spreadsheets",
        "coding", "linux",
        "spreadsheets", "organization"
    ]
})
display(df_1, df_6)

In [ ]:
pd.merge(df_1, df_6, on='group')

<hr style="border: none; height: 5px; background-color: gray;">

### Different key names: `left_on` / `right_on`
Useful when the key column names differ.

In [ ]:
df_7 = pd.DataFrame({
    "name": ["Bob", "Jake", "Lisa", "Sue"],
    "salary": [70000, 80000, 120000, 90000]
})

display(df_1, df_7)

In [ ]:
pd.merge(df_1, df_7, left_on='employee', right_on='name')

In [ ]:
# Drop redundant columns if needed
df = pd.merge(df_1, df_7, left_on='employee', right_on='name')
df.drop('name', axis=1)

<hr style="border: none; height: 5px; background-color: gray;">

### Merge strategies with `how=`
- `inner`: keep only matches
- `outer`: keep all rows from both sides
- `left`: keep all left rows
- `right`: keep all right rows

In [ ]:
df_8 = pd.DataFrame({
    "employee": ["Bob", "Jake", "Lisa"],
    "salary": [70000, 80000, 120000]
})
display(df_1, df_8)

In [ ]:
pd.merge(df_1, df_8, on='employee', how='left')

<hr style="border: none; height: 10px; background-color: LightBlue;">

## `.join()` merges on index by default
Use `.join()` when your identifiers are already in the index.

In [ ]:
df1 = pd.DataFrame({
    "group": ["Accounting", "Engineering", "Engineering", "HR"]
}, index=["Bob", "Jake", "Lisa", "Sue"])

df2 = pd.DataFrame({
    "hire_date": [2004, 2008, 2012, 2014]
}, index=["Lisa", "Bob", "Jake", "Sue"])
display(df1, df2)

In [ ]:
df1.join(df2)